In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from PIL import Image, ImageTk
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
import os
import qrcode
import time


SECRET_KEY = b"mysecretpassword"
encoded_message = None  

def encrypt_message(message):
    cipher = AES.new(SECRET_KEY, AES.MODE_CBC)
    ciphertext = cipher.encrypt(pad(message.encode(), AES.block_size))
    return cipher.iv + ciphertext


def decrypt_message(ciphertext):
    iv = ciphertext[:16]
    cipher = AES.new(SECRET_KEY, AES.MODE_CBC, iv)
    plaintext = unpad(cipher.decrypt(ciphertext[16:]), AES.block_size)
    return plaintext.decode()



def read_bmp(file_path):
    with open(file_path, "rb") as f:
        data = f.read()
    header = data[:54]  
    pixels = bytearray(data[54:])
    return header, pixels


def write_bmp(file_path, header, pixels):
    with open(file_path, "wb") as f:
        f.write(header)
        f.write(pixels)


def message_to_binary(message):
    return ''.join(format(byte, '08b') for byte in message)


def binary_to_message(binary_str):
    binary_chunks = [binary_str[i:i + 8] for i in range(0, len(binary_str), 8)]
    return bytearray(int(chunk, 2) for chunk in binary_chunks)


def encode_message_in_bmp(input_path, output_path, message):
    header, pixels = read_bmp(input_path)
    binary_message = message_to_binary(message) + '1111111111111110' 
    data_index = 0

    for i in range(len(pixels)):
        if data_index < len(binary_message):
            pixels[i] = (pixels[i] & 0xFE) | int(binary_message[data_index])
            data_index += 1

    write_bmp(output_path, header, pixels)


def decode_message_from_bmp(input_path):
    _, pixels = read_bmp(input_path)
    binary_message = ""

    for pixel in pixels:
        binary_message += str(pixel & 1)

    eof_index = binary_message.find('1111111111111110')
    if eof_index != -1:
        binary_message = binary_message[:eof_index]
    else:
        raise ValueError("No hidden message found.")

    return binary_to_message(binary_message)


# Animation Functions
def fade_in(widget, duration=500):
    for i in range(0, 101, 5):
        alpha = i / 100
        root.attributes("-alpha", alpha)
        widget.update()
        time.sleep(duration / 5000)


def fade_out(widget, duration=500):
    for i in range(100, -1, -5):  
        alpha = i / 100
        root.attributes("-alpha", alpha)
        widget.update()
        time.sleep(duration / 5000)


def progress_animation(progress_bar):
    for _ in range(30):
        progress_bar.step(3.33)
        progress_bar.update_idletasks()
        time.sleep(0.03)


def animate_button_hover(button, on_hover=True):
    def smooth_transition():
        current_color = button.cget("bg")
        target_color = "#2980b9" if on_hover else "#3498db"
        steps = 10
        r, g, b = [int(current_color[i:i+2], 16) for i in (1, 3, 5)]
        tr, tg, tb = [int(target_color[i:i+2], 16) for i in (1, 3, 5)]
        for i in range(steps):
            r = r + (tr - r) // steps
            g = g + (tg - g) // steps
            b = b + (tb - b) // steps
            button.config(bg=f"#{r:02x}{g:02x}{b:02x}")
            button.after(50)

    smooth_transition()


def animate_focus(entry, on_focus=True):
    if on_focus:
        entry.config(bg="#eaf2f8")  
    else:
        entry.config(bg="white")  


def select_file():
    file_path = filedialog.askopenfilename(filetypes=[("BMP Files", "*.bmp")])
    if not file_path:
        return
    entry_file_path.delete(0, tk.END)
    entry_file_path.insert(0, file_path)
    display_image(file_path, "Original Image")


def display_image(file_path, title):
    img = Image.open(file_path)
    img.thumbnail((400, 400))
    img_tk = ImageTk.PhotoImage(img)

    label_img.configure(image=img_tk)
    label_img.image = img_tk
    label_img_text.config(text=title)


def encode_message():
    input_path = entry_file_path.get()
    message = entry_message.get()
    if not input_path or not message:
        messagebox.showerror("Error", "Please select a file and enter a message.")
        return

    output_path = input_path.replace(".bmp", "_stego.bmp")
    progress_animation(progress) 

    try:
        encrypted_message = encrypt_message(message)
        encode_message_in_bmp(input_path, output_path, encrypted_message)
        messagebox.showinfo("Success", f"Message encoded successfully in {output_path}")
        display_image(output_path, "Encoded Image")
    except Exception as e:
        messagebox.showerror("Error", f"Failed to encode message: {e}")


def decode_message():
    input_path = entry_file_path.get()
    if not input_path:
        messagebox.showerror("Error", "Please select a file.")
        return

    try:
        message = decode_message_from_bmp(input_path)
        decrypted_message = decrypt_message(message)
        messagebox.showinfo("Decoded Message", f"The hidden message is:\n\n{decrypted_message}")
    except Exception as e:
        messagebox.showerror("Error", f"Failed to decode message: {e}")


def view_binary_message():
    message = entry_message.get()
    if not message:
        messagebox.showerror("Error", "Please enter a message.")
        return

    encrypted_message = encrypt_message(message)
    binary_message = message_to_binary(encrypted_message) + " (EOF: 1111111111111110)"
    text_binary_output.delete(1.0, tk.END)
    text_binary_output.insert(tk.END, binary_message)


def generate_qr_code():
    message = entry_message.get()
    if not message:
        messagebox.showerror("Error", "Please enter a message.")
        return

    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(message)
    qr.make(fit=True)

    img = qr.make_image(fill="black", back_color="white")
    qr_path = "qr_code.png"
    img.save(qr_path)

    display_image(qr_path, "Generated QR Code")
    messagebox.showinfo("QR Code", f"QR Code generated and saved as {qr_path}")


def clear_fields():
    entry_file_path.delete(0, tk.END)
    entry_message.delete(0, tk.END)
    text_binary_output.delete(1.0, tk.END)
    label_img.config(image="")
    label_img_text.config(text="No Image Selected")



root = tk.Tk()
root.title("Advanced Image Steganography with Features")
root.geometry("800x650")
root.config(bg="#f4f4f4")


fade_in(root)


frame_file = tk.Frame(root, bg="#f4f4f4")
frame_file.pack(pady=10)
label_file_path = tk.Label(frame_file, text="Image File:", font=("Helvetica", 14), bg="#f4f4f4")
label_file_path.pack(side=tk.LEFT, padx=5)
entry_file_path = tk.Entry(frame_file, width=60, font=("Helvetica", 12))
entry_file_path.pack(side=tk.LEFT, padx=5)
button_browse = tk.Button(frame_file, text="Browse", font=("Helvetica", 12), command=select_file, bg="#3498db", fg="white")
button_browse.pack(side=tk.LEFT, padx=5)


label_img_text = tk.Label(root, text="No Image Selected", font=("Helvetica", 14), bg="#f4f4f4")
label_img_text.pack(pady=10)
label_img = tk.Label(root, bg="#f4f4f4")
label_img.pack()


frame_message = tk.Frame(root, bg="#f4f4f4")
frame_message.pack(pady=10)
label_message = tk.Label(frame_message, text="Secret Message:", font=("Helvetica", 14), bg="#f4f4f4")
label_message.pack(side=tk.LEFT, padx=5)
entry_message = tk.Entry(frame_message, width=60, font=("Helvetica", 12))
entry_message.pack(side=tk.LEFT, padx=5)


progress = ttk.Progressbar(root, orient="horizontal", length=300, mode="indeterminate")
progress.pack(pady=10)

frame_binary = tk.Frame(root, bg="#f4f4f4")
frame_binary.pack(pady=10)
button_view_binary = tk.Button(frame_binary, text="View Binary Message", font=("Helvetica", 12), command=view_binary_message, bg="#3498db", fg="white")
button_view_binary.pack(side=tk.LEFT, padx=5)
text_binary_output = tk.Text(frame_binary, height=5, width=60, font=("Courier New", 10), wrap=tk.WORD)
text_binary_output.pack(side=tk.LEFT, padx=5)


frame_buttons = tk.Frame(root, bg="#f4f4f4")
frame_buttons.pack(pady=10)

button_encode = tk.Button(frame_buttons, text="Encode", font=("Helvetica", 14), command=encode_message, bg="#3498db", fg="white")
button_encode.grid(row=0, column=0, padx=20)
button_decode = tk.Button(frame_buttons, text="Decode", font=("Helvetica", 14), command=decode_message, bg="#3498db", fg="white")
button_decode.grid(row=0, column=1, padx=20)
button_qr = tk.Button(frame_buttons, text="Generate QR Code", font=("Helvetica", 14), command=generate_qr_code, bg="#3498db", fg="white")
button_qr.grid(row=0, column=2, padx=20)
button_clear = tk.Button(frame_buttons, text="Clear", font=("Helvetica", 14), command=clear_fields, bg="#95a5a6", fg="white")
button_clear.grid(row=0, column=3, padx=20)


for button in [button_encode, button_decode, button_qr, button_clear]:
    button.bind("<Enter>", lambda event, b=button: animate_button_hover(b, on_hover=True))
    button.bind("<Leave>", lambda event, b=button: animate_button_hover(b, on_hover=False))

entry_file_path.bind("<FocusIn>", lambda event, e=entry_file_path: animate_focus(e, on_focus=True))
entry_file_path.bind("<FocusOut>", lambda event, e=entry_file_path: animate_focus(e, on_focus=False))
entry_message.bind("<FocusIn>", lambda event, e=entry_message: animate_focus(e, on_focus=True))
entry_message.bind("<FocusOut>", lambda event, e=entry_message: animate_focus(e, on_focus=False))

root.mainloop()
